In [1]:

import numpy as np
import pandas as pd
from pathlib import Path

# 1. Configuration
SEED = 42
N = 10_000
rng = np.random.default_rng(SEED)

# 2. Customer history and profile
df = pd.DataFrame({
    "customer_id": [f"C{i:05d}" for i in range(1, N + 1)],
    "tenure_days": rng.integers(30, 2001, N),
    "previous_orders": rng.poisson(8, N),
    "days_since_last_order": rng.integers(1, 181, N),
    "avg_order_value": np.round(
        rng.lognormal(6.5, 0.45, N), 2
    ),
    "age": rng.integers(18, 71, N),
    "customer_segment": rng.choice(
        ["New", "Regular", "Premium"], N,
        p=[0.25, 0.50, 0.25]
    ),
    "region": rng.choice(
        ["North", "South", "East", "West"], N
    ),
    "membership_tier": rng.choice(
        ["Standard", "Silver", "Gold"], N,
        p=[0.50, 0.30, 0.20]
    ),
    "prior_email_engagement": np.round(
        rng.beta(2, 3, N), 3
    ),
    "email_clicks_90d": rng.poisson(3, N),
    "website_sessions_30d": rng.poisson(6, N),
    "cart_additions_30d": rng.poisson(2, N),
    "price_sensitivity": np.round(
        rng.beta(2, 2, N), 3
    ),
    "campaign_id": rng.choice(
        ["CMP001", "CMP002", "CMP003", "CMP004"], N
    ),
    "campaign_channel": rng.choice(
        ["Email", "SMS", "Push", "Web"], N
    ),
    "list_price": np.round(
        rng.uniform(500, 5000, N), 2
    )
})

# 3. Product economics
df["unit_cost"] = np.round(
    df["list_price"] * rng.uniform(0.40, 0.75, N), 2
)

# 4. Assign discounts based on customer characteristics
# This intentionally creates confounding.
def standardize(series):
    return (series - series.mean()) / (series.std() + 1e-8)

tenure_z = standardize(df["tenure_days"]).to_numpy()
orders_z = standardize(df["previous_orders"]).to_numpy()
recency_z = standardize(df["days_since_last_order"]).to_numpy()

engagement = df["prior_email_engagement"].to_numpy()
sensitivity = df["price_sensitivity"].to_numpy()

offer_score = (
    -0.3
    + 0.25 * tenure_z
    - 0.30 * orders_z
    + 0.45 * recency_z
    + 0.60 * sensitivity
    + 0.30 * (1 - engagement)
)

p_offer = 1 / (1 + np.exp(-offer_score))
offered = rng.binomial(1, p_offer)

discount = np.zeros(N)
discount[offered == 1] = rng.choice(
    [10, 20], size=offered.sum(), p=[0.60, 0.40]
)

df["discount_percent"] = discount
df["treatment"] = (discount > 0).astype(int)

# 5. Generate purchase probability
baseline_log_odds = (
    -0.8
    + 0.25 * orders_z
    - 0.25 * recency_z
    + 0.80 * engagement
    + 0.35 * sensitivity
    + 0.15 * np.log1p(
        df["website_sessions_30d"].to_numpy()
    )
)

discount_effect = (
    0.035 * discount
    + 0.025 * discount * sensitivity
)

p_with_discount = 1 / (
    1 + np.exp(-(baseline_log_odds + discount_effect))
)
p_without_discount = 1 / (
    1 + np.exp(-baseline_log_odds)
)

df["purchased"] = rng.binomial(1, p_with_discount)
df["purchase_probability"] = np.round(p_with_discount, 5)

# Evaluation-only field: exclude from model inputs
df["true_treatment_effect"] = np.round(
    p_with_discount - p_without_discount, 5
)

# 6. Revenue and profit
realized_price = (
    df["list_price"] * (1 - discount / 100)
)

df["revenue"] = np.round(
    df["purchased"] * realized_price, 2
)
df["discount_cost"] = np.round(
    df["purchased"] * df["list_price"] * discount / 100,
    2
)
df["profit"] = np.round(
    df["revenue"] - df["purchased"] * df["unit_cost"], 2
)

# 7. Save CSV
output_path = Path("data") / "synthetic_campaign_data.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)

# 8. Validate
print("Dataset generated successfully")
print("Shape:", df.shape)
print("Saved to:", output_path)
print("Missing values:", df.isnull().sum().sum())

print("\nDiscount distribution:")
print(df["discount_percent"].value_counts().sort_index())

print("\nPurchase rate by discount:")
print(df.groupby("discount_percent")["purchased"].mean().round(3))

print("\nFirst five rows:")
print(df.head())

Dataset generated successfully
Shape: (10000, 26)
Saved to: data/synthetic_campaign_data.csv
Missing values: 0

Discount distribution:
discount_percent
0.0     4625
10.0    3211
20.0    2164
Name: count, dtype: int64

Purchase rate by discount:
discount_percent
0.0     0.507
10.0    0.592
20.0    0.701
Name: purchased, dtype: float64

First five rows:
  customer_id  tenure_days  previous_orders  days_since_last_order  \
0      C00001          205                6                     73   
1      C00002         1555                8                     98   
2      C00003         1320               13                     28   
3      C00004          895                2                     96   
4      C00005          883                8                    151   

   avg_order_value  age customer_segment region membership_tier  \
0           428.78   29          Premium  North        Standard   
1           507.29   42              New  South            Gold   
2           634.04   27 

In [2]:
from google.colab import files

files.download("data/synthetic_campaign_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
print("Rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nTreatment counts:")
print(df["treatment"].value_counts())

print("\nPurchase rate:")
print(df["purchased"].mean())

Rows and columns: (10000, 26)

Column names:
['customer_id', 'tenure_days', 'previous_orders', 'days_since_last_order', 'avg_order_value', 'age', 'customer_segment', 'region', 'membership_tier', 'prior_email_engagement', 'email_clicks_90d', 'website_sessions_30d', 'cart_additions_30d', 'price_sensitivity', 'campaign_id', 'campaign_channel', 'list_price', 'unit_cost', 'discount_percent', 'treatment', 'purchased', 'purchase_probability', 'true_treatment_effect', 'revenue', 'discount_cost', 'profit']

Treatment counts:
treatment
1    5375
0    4625
Name: count, dtype: int64

Purchase rate:
0.5766


In [4]:
from google.colab import files

uploaded = files.upload()

Saving synthetic_campaign_data.csv to synthetic_campaign_data.csv


In [5]:
import pandas as pd

df = pd.read_csv("synthetic_campaign_data.csv")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Dataset shape: (10000, 26)
Columns: ['customer_id', 'tenure_days', 'previous_orders', 'days_since_last_order', 'avg_order_value', 'age', 'customer_segment', 'region', 'membership_tier', 'prior_email_engagement', 'email_clicks_90d', 'website_sessions_30d', 'cart_additions_30d', 'price_sensitivity', 'campaign_id', 'campaign_channel', 'list_price', 'unit_cost', 'discount_percent', 'treatment', 'purchased', 'purchase_probability', 'true_treatment_effect', 'revenue', 'discount_cost', 'profit']


,customer_id,tenure_days,previous_orders,days_since_last_order,avg_order_value,age,customer_segment,region,membership_tier,prior_email_engagement,...,list_price,unit_cost,discount_percent,treatment,purchased,purchase_probability,true_treatment_effect,revenue,discount_cost,profit
0,C00001,205,6,73,428.78,29,Premium,North,Standard,0.235,...,2472.74,1605.75,20.0,1,1,0.63060,0.21083,1978.19,494.55,372.44
1,C00002,1555,8,98,507.29,42,New,South,Gold,0.408,...,3453.33,1789.85,0.0,0,1,0.47824,0.00000,3453.33,0.00,1663.48
2,C00003,1320,13,28,634.04,27,New,North,Standard,0.481,...,2635.29,1865.58,0.0,0,0,0.68586,0.00000,0.00,0.00,0.00
3,C00004,895,2,96,570.69,20,Regular,South,Gold,0.192,...,2089.68,1342.04,20.0,1,0,0.51868,0.20624,0.00,0.00,0.00
4,C00005,883,8,151,453.66,70,Regular,West,Gold,0.201,...,865.00,410.61,20.0,1,0,0.54148,0.18924,0.00,0.00,0.00


In [6]:
print(df.columns.tolist())

['customer_id', 'tenure_days', 'previous_orders', 'days_since_last_order', 'avg_order_value', 'age', 'customer_segment', 'region', 'membership_tier', 'prior_email_engagement', 'email_clicks_90d', 'website_sessions_30d', 'cart_additions_30d', 'price_sensitivity', 'campaign_id', 'campaign_channel', 'list_price', 'unit_cost', 'discount_percent', 'treatment', 'purchased', 'purchase_probability', 'true_treatment_effect', 'revenue', 'discount_cost', 'profit']


In [7]:
print("Treatment values:")
print(df["treatment"].value_counts())

print("\nOutcome values:")
print(df["purchased"].value_counts())

print("\nMissing values:")
print(df[["treatment", "purchased"]].isnull().sum())

Treatment values:
treatment
1    5375
0    4625
Name: count, dtype: int64

Outcome values:
purchased
1    5766
0    4234
Name: count, dtype: int64

Missing values:
treatment    0
purchased    0
dtype: int64


In [11]:
from dowhy import CausalModel

causal_graph = """
digraph {
    tenure_days -> treatment;
    tenure_days -> purchased;

    previous_orders -> treatment;
    previous_orders -> purchased;

    days_since_last_order -> treatment;
    days_since_last_order -> purchased;

    prior_email_engagement -> treatment;
    prior_email_engagement -> purchased;

    price_sensitivity -> treatment;
    price_sensitivity -> purchased;

    treatment -> purchased;
    treatment -> discount_percent;

    discount_percent -> purchased;
    avg_order_value -> revenue;
    discount_percent -> revenue;
    purchased -> revenue;
}
"""

model = CausalModel(
    data=df,
    treatment="treatment",
    outcome="purchased",
    graph=causal_graph
)

print("Causal model created successfully.")

ERROR:dowhy.causal_graph:Error: Pygraphviz cannot be loaded. No module named 'pygraphviz'
Trying pydot ...


Causal model created successfully.


In [12]:
# Step 1: Identify the causal effect
identified_estimand = model.identify_effect(
    estimand_type="nonparametric-ate"
)

print("Identified estimand:")
print(identified_estimand)

Identified estimand:
Estimand type: EstimandType.NONPARAMETRIC_ATE

### Estimand : 1
Estimand name: backdoor
Estimand expression:
     d                                                                         ↪
────────────(E[purchased|prior_email_engagement,days_since_last_order,tenure_d ↪
d[treatment]                                                                   ↪

↪                                        
↪ ays,previous_orders,price_sensitivity])
↪                                        
Estimand assumption 1, Unconfoundedness: If U→{treatment} and U→purchased then P(purchased|treatment,prior_email_engagement,days_since_last_order,tenure_days,previous_orders,price_sensitivity,U) = P(purchased|treatment,prior_email_engagement,days_since_last_order,tenure_days,previous_orders,price_sensitivity)

### Estimand : 2
Estimand name: iv
No such variable(s) found!

### Estimand : 3
Estimand name: frontdoor
No such variable(s) found!

### Estimand : 4
Estimand name: general_adjustment
Esti

In [13]:
# Step 2: Estimate the average treatment effect
estimate = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.linear_regression"
)

print("Estimated causal effect:", estimate.value)

Estimated causal effect: 0.17066051842684132


In [14]:
refutation = model.refute_estimate(
    identified_estimand,
    estimate,
    method_name="random_common_cause"
)

print(refutation)

Refute: Add a random common cause
Estimated effect:0.17066051842684132
New effect:0.1706621221457768
p value:0.9



In [10]:
%pip install dowhy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 65.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 113.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 70.1 MB/s  0:00:00
  Attempting uninstall: cvxpy
    Found existing installation: cvxpy 1.6.7
    Uninstalling cvxpy-1.6.7:
      Successfully uninstalled cvxpy-1.6.7
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [dowhy]
